In [25]:
"""
Импорт необходимых библиотек.
pandas - для работы с табличными данными.
numpy - для работы с массивами.
faiss - для индексации и поиска векторов.
SentenceTransformer - для генерации текстовых эмбеддингов.
tqdm - для отображения прогресса выполнения операций.
"""
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

tqdm.pandas()

In [26]:
"""
Загрузка очищенных данных.
Предполагаем, что файл data/movies_cleaned.csv существует.
Заполняем пропуски в столбце text_features пустыми строками, чтобы избежать ошибок при кодировании.
"""
df = pd.read_csv('data/movies_cleaned.csv')
df['text_features'] = df['text_features'].fillna('')

In [27]:
"""
Инициализация модели для создания эмбеддингов.
Используется модель all-MiniLM-L6-v2, которая обеспечивает хороший баланс между скоростью и качеством.
"""
model = SentenceTransformer('all-MiniLM-L6-v2')

In [28]:
"""
Генерация эмбеддингов.
Преобразуем текстовые признаки в векторы для всех фильмов.
Результат конвертируем в массив numpy с типом float32, так как FAISS работает именно с этим типом данных.
"""
embeddings = model.encode(df['text_features'].tolist(), show_progress_bar=True)
embeddings = np.array(embeddings).astype('float32')

Batches:   0%|          | 0/1458 [00:00<?, ?it/s]

In [29]:
"""
Создание и настройка индекса FAISS.
Сначала нормализуем векторы (L2-нормализация).
Это необходимо для того, чтобы IndexFlatIP вычислял косинусное сходство.
Создаем плоский индекс на основе внутреннего произведения (Inner Product).
Добавляем векторы в индекс.
"""
faiss.normalize_L2(embeddings)

dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)

index.add(embeddings)

In [30]:
"""
Сохранение индекса на диск.
Индекс сохраняется под именем data/movies_faiss.index.
"""
faiss.write_index(index, 'data/movies_faiss.index')

In [31]:
"""
Функция поиска похожих фильмов.
Принимает текст запроса и количество желаемых результатов (по умолчанию 5).
Процесс:
1. Кодирование текста запроса.
2. Приведение к float32 и нормализация (L2).
3. Поиск ближайших векторов в индексе.
4. Возврат списка словарей с названиями и описаниями найденных фильмов.
"""
def search_similar_movies(query_text, top_k=5):
    query_embedding = model.encode([query_text])
    query_embedding = np.array(query_embedding).astype('float32')
    
    faiss.normalize_L2(query_embedding)
    
    distances, indices = index.search(query_embedding, top_k)
    
    """ Формируем список результатов, извлекая title и overview из датафрейма """
    found_movies = []
    for idx in indices[0]:
        title = df.iloc[idx]['title']
        overview = df.iloc[idx]['overview']
        found_movies.append({'title': title, 'overview': overview})
        
    return found_movies

In [32]:
"""
Тестирование функции поиска.
В качестве запроса используется фраза: Мрачный детектив про серийных убийц.
Результат выводится в консоль с форматированием.
"""
query = "A dark detective story about serial killers"
results = search_similar_movies(query)

print(f"Результаты поиска для запроса: '{query}'\n")
print("=" * 70)

for i, movie in enumerate(results, 1):
    print(f"{i}. Название: {movie['title']}")
    
    """ 
    Обрезаем слишком длинные описания для аккуратного вывода.
    Приводим к строке на случай, если попался NaN, который не отловился ранее.
    """
    overview = str(movie['overview'])
    if len(overview) > 250:
        overview = overview[:247] + "..."
        
    print(f"   Описание: {overview}")
    print("-" * 70)

Результаты поиска для запроса: 'A dark detective story about serial killers'

1. Название: So Sweet, So Dead
   Описание: A serial killer is on the loose. His victims are unfaithful wives and he always leaves compromising photographs at the crime scene.
----------------------------------------------------------------------
2. Название: Zodiac
   Описание: The true story of the investigation of 'The Zodiac Killer',  a serial killer who terrified the San Francisco Bay Area, taunting police with his ciphers and letters. The case becomes an obsession for four men as their lives and careers are built an...
----------------------------------------------------------------------
3. Название: Brutal
   Описание: In a small town, a serial killer mutilates the bodies of his victims and leaves a flower on the corpses. The sheriff and his wife/deputy investigate the murders while trying to keep from alarming the citizens. They team up with an autistic hound d...
------------------------------------